In [3]:
!pip install -q transformers accelerate sentencepiece
import faiss
print(faiss.__version__)

ERROR: Could not find a version that satisfies the requirement faiss (from versions: none)
ERROR: No matching distribution found for faiss


ModuleNotFoundError: No module named 'faiss'

In [ ]:
import torch
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    CLIPProcessor,
    CLIPModel,
    pipeline
)

import faiss

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

In [ ]:
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

In [ ]:
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

In [ ]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto"
)

In [ ]:
def generate_caption(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = blip_processor(
        image,
        return_tensors="pt"
    ).to(device)

    output = blip_model.generate(
        **inputs,
        max_new_tokens=40
    )

    caption = blip_processor.decode(
        output[0],
        skip_special_tokens=True
    )

    return caption

In [ ]:
def generate_metadata(caption):

    prompt = f"""
Convert this product description into ecommerce metadata.

Product:
{caption}

Output format:

Category:
Title:
Description:
Tags:
"""
    result = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        temperature=0.1
    )

    return result[0]["generated_text"]

In [ ]:
image_path = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small/images/10005.jpg"

caption = generate_caption(image_path)

print("CAPTION:")
print(caption)

In [ ]:
metadata = generate_metadata(caption)

print(metadata)

In [ ]:
fields = [
    "Category:",
    "Title:",
    "Description:",
    "Tags:",
    "Color:",
]

lines = metadata.split("\n")

clean_lines = []

for line in lines:
    if any(line.startswith(field) for field in fields):
        clean_lines.append(line)

print("\n".join(clean_lines))

In [ ]:
import os

image_dir = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small/images"

all_images = [
    os.path.join(image_dir, x)
    for x in os.listdir(image_dir)
][:100]

In [ ]:
from transformers import AutoImageProcessor, AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(
    "google/vit-base-patch16-224-in21k"
)

vit_model = AutoModel.from_pretrained(
    "google/vit-base-patch16-224-in21k"
).to(device)

In [ ]:
from PIL import Image
import numpy as np

def get_image_embedding(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = vit_model(**inputs)

        embedding = outputs.pooler_output

    embedding = embedding.detach().cpu().numpy()

    embedding = embedding / np.linalg.norm(
        embedding,
        axis=1,
        keepdims=True
    )

    return embedding

In [ ]:
emb = get_image_embedding(all_images[0])

print(type(emb))
print(emb.shape)

In [ ]:
all_images = all_images[:50]

In [ ]:
from tqdm import tqdm

embeddings = []

for img_path in tqdm(all_images):

    emb = get_image_embedding(img_path)

    embeddings.append(emb[0])

embeddings = np.array(embeddings)

print(embeddings.shape)

In [ ]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

In [ ]:
def search_similar_products(query_image, k=5):

    query_embedding = get_image_embedding(query_image)

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    return indices[0], scores[0]

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

query_image = all_images[0]

indices, scores = search_similar_products(
    query_image,
    k=6
)

fig = plt.figure(figsize=(15,8))

# Query Image
plt.subplot(2, 6, 3)
img = Image.open(query_image)
plt.imshow(img)
plt.axis("off")
plt.title("QUERY IMAGE", fontsize=12)

# Similar Images
for i in range(1, len(indices)):

    idx = indices[i]

    plt.subplot(2, 5, i + 5)

    sim_img = Image.open(all_images[idx])

    plt.imshow(sim_img)
    plt.axis("off")

    plt.title(
        f"Score: {scores[i]:.2f}",
        fontsize=10
    )

plt.tight_layout()
plt.show()